# Suite TSDEF — TypeScript definitions

The names a TypeScript or JavaScript program declares, in the scopes ECMAScript and TypeScript give them
(`TypeScript.Definitions.define`): one entity per name and kind per scope, merging what TypeScript merges, and
lookup outward that keeps values, types and namespaces apart. Each case checks the entities, their declarations,
and what names resolve to.

In [ ]:
from pathlib import Path

from mbse.Programs.Framework import Syntax as F
from mbse.Programs.TypeScript import Definitions as D, Syntax as S, TypeScript59

TS = TypeScript59.STANDARD
SOURCES = Path("../../conformance/sources")


def define(text, standard=TS):
    program = standard.parse(text)
    return program, D.define(program)


def listing(program):
    return [(e.kind, e.qualified_name(), len(e.declarations)) for e in program.entities()]


def names(tree, name):
    """The identifiers of `tree` named `name` that are references (not declarations), in source order."""
    return [n for n in F.walk(tree) if isinstance(n, (S.Identifier, S.JSXIdentifier)) and n.name == name]


def refers(program, node):
    return [f"{e.kind}:{e.qualified_name()}" for e in D.referents(program, node)]

## TSDEF-01 · What declarations declare, and which merge: one entity per name and kind per scope

In [ ]:
tree, program = define("""
var v = 1; var v;
let [a, , ...b] = c, { d, e: f = 1, ...g } = h;
const k = 1;
function over(x: string): string;
function over(x: any) { return x; }
interface I { a: string }
interface I { b: number }
namespace N { export const x = 1; }
namespace N { export const y = 2; }
enum E { A }
enum E { B = 1 }
type T = string;
class C {}
import def, { one as two, type three } from "m";
import * as all from "m";
import eq = require("m");
declare function amb(): void;
export default function () {}
""")
assert listing(program) == [
    ("variable", "v", 2), ("variable", "a", 1), ("variable", "b", 1), ("variable", "d", 1), ("variable", "f", 1),
    ("variable", "g", 1), ("variable", "k", 1), ("function", "over", 2), ("parameter", "over.x", 1),
    ("parameter", "over.x", 1), ("interface", "I", 2), ("property", "I.a", 1), ("property", "I.b", 1),
    ("namespace", "N", 2), ("variable", "N.x", 1), ("variable", "N.y", 1), ("enum", "E", 2),
    ("enum member", "E.A", 1), ("enum member", "E.B", 1), ("type alias", "T", 1), ("class", "C", 1),
    ("import", "def", 1), ("import", "two", 1), ("import", "three", 1), ("import", "all", 1), ("import", "eq", 1),
    ("function", "amb", 1)]
over = next(e for e in program.entities() if e.name == "over")
assert [type(d).__name__ for d in over.declarations] == ["TSDeclareFunction", "FunctionDeclaration"]
assert isinstance(over.definition, S.FunctionDeclaration)
assert over.scope.node is over.definition  # an overloaded function's scope is its implementation's
amb = next(e for e in program.entities() if e.name == "amb")
assert amb.definition is None and amb.scope is not None
interface = next(e for e in program.entities() if e.kind == "interface")
assert interface.definition is interface.declarations[0] and interface.scope.kind == "interface"
assert program.lookup("I.a")[0].definition is None  # a signature declares, and defines nothing
assert program.root.separator == "." and program.lookup("N.y")[0].kind == "variable"
assert [e.kind for e in program.lookup("E")] == ["enum"] and program.lookup("E.B")[0].parent.kind == "enum"
print(len(listing(program)), "entities")

## TSDEF-02 · Scopes: `var` in its function, the rest in their blocks; loops, switches, catches and static blocks

In [ ]:
tree, program = define("""
function f(p: number) {
    if (p) { var hoisted = 1; let local = 2; }
    for (let i = 0; i < p; i++) { const j = i; }
    for (const key in o) {}
    for (const value of o) {}
    switch (p) { case 1: let s = 1; }
    try {} catch ({ message }) { message; }
    label: for (;;) { break label; }
    return hoisted + i;
}
class K { static { var inStatic = 1; } }
const fe = function named() { return named; };
const arrow = (a = b, { c } = a) => a + c;
const ce = class Named { m() { return Named; } };
const { [key]: renamed } = o;
const anonymous = class {};
""")
assert listing(program) == [
    ("function", "f", 1), ("parameter", "f.p", 1), ("variable", "f.hoisted", 1), ("variable", "local", 1),
    ("variable", "i", 1), ("variable", "j", 1), ("variable", "key", 1), ("variable", "value", 1),
    ("variable", "s", 1), ("variable", "message", 1), ("class", "K", 1), ("variable", "inStatic", 1),
    ("variable", "fe", 1), ("function", "named", 1), ("variable", "arrow", 1), ("parameter", "a", 1),
    ("parameter", "c", 1), ("variable", "ce", 1), ("class", "Named", 1), ("method", "Named.m", 1),
    ("variable", "renamed", 1), ("variable", "anonymous", 1)]
key = names(tree, "key")[-1]  # the pattern's computed key, outside the loop whose `key` it is
assert program.scope_of(key) is program.root and refers(program, key) == []
scopes = {e.name: e.parent.kind for e in program.entities()}
assert program.lookup("Named") == [] and scopes["Named"] == "block"  # a class expression's name is its own
assert (scopes["local"], scopes["i"], scopes["s"], scopes["message"], scopes["inStatic"]) == (
    "block", "block", "block", "block", "static block")
hoisted, i = names(tree, "hoisted")[-1], names(tree, "i")[-1]
assert refers(program, hoisted) == ["variable:f.hoisted"] and refers(program, i) == []  # `i` is the loop's
assert refers(program, names(tree, "named")[-1]) == ["function:named"]
assert refers(program, names(tree, "Named")[-1]) == ["class:Named"]
assert refers(program, names(tree, "b")[0]) == [] and refers(program, names(tree, "a")[-1]) == ["parameter:a"]
assert program.scope_of(names(tree, "label")[1]) is None  # a label names no entity
assert refers(program, names(tree, "message")[-1]) == ["variable:message"]
print("ok")

## TSDEF-03 · Classes and interfaces: members, accessors, private names, parameter properties and heritage

In [ ]:
tree, program = define("""
abstract class Box<T> extends Base<T> implements Shape {
    #secret = 1;
    static count = 0;
    accessor size = 1;
    [computed] = 2;
    'quoted' = 3;
    "double" = 4;
    0 = 5;
    constructor(private readonly value: T, public label = "x", plain: number) { super(); }
    get area(): number { return this.#secret; }
    set area(v: number) {}
    abstract run(): void;
    method?(): void;
    method?(): void {}
    [key: string]: unknown;
    @decorated field = Box.count;
}
interface Shape<T = number> extends Base<T> {
    area: number; move(): void; get size(): number; [k: string]: unknown; [Symbol.iterator](): void
}
""")
box = next(e for e in program.entities() if e.name == "Box")
assert [(e.kind, e.name, len(e.declarations)) for e in box.scope.entities()] == [
    ("property", "#secret", 1), ("property", "count", 1), ("accessor", "size", 1), ("property", "quoted", 1),
    ("property", "double", 1), ("property", "0", 1), ("method", "constructor", 1), ("property", "value", 1),
    ("property", "label", 1),
    ("accessor", "area", 2),
    ("method", "run", 1), ("method", "method", 2), ("property", "field", 1)]
assert box.scope.parent.kind == "type parameters"
assert refers(program, names(tree, "T")[1]) == ["type parameter:T"]  # `Base<T>`
assert refers(program, names(tree, "Base")[0]) == [] and refers(program, names(tree, "Shape")[0]) == ["interface:Shape"]
assert refers(program, names(tree, "computed")[0]) == []
assert refers(program, names(tree, "count")[1]) == []  # a property's name depends on the value
assert refers(program, names(tree, "Box")[1]) == ["class:Box"]
constructor = next(e for e in box.scope.entities() if e.name == "constructor")
assert [e.kind for e in constructor.scope.entities()] == ["parameter", "parameter", "parameter"]
method = next(e for e in box.scope.entities() if e.name == "method")
assert isinstance(method.definition.value, S.FunctionExpression)
shape = next(e for e in program.entities() if e.kind == "interface")
assert [(e.kind, e.name) for e in shape.scope.entities()] == [("property", "area"), ("method", "move"),
                                                             ("accessor", "size")]
assert program.lookup("Box.count")[0].kind == "property"
print("ok")

## TSDEF-04 · Meanings: a name is a value, a type or a namespace where it is written, and finds only those

In [ ]:
tree, program = define("""
type Point = { x: number };
const Point = 1;
namespace Geo { export interface Shape {} }
let p: Point = Point;
let q: typeof Point;
let s: Geo.Shape;
let g = Geo;
let a = <Point>p, b = p as Point, c = p satisfies Point, d = [p] as const;
function guard(x: unknown): x is Point { return true; }
export { Point };
export { Point as Other } from "m";
const Point2 = Point;
""")
point = names(tree, "Point")
spaces = [program.space_of(n) for n in point]
assert spaces == [None, None, "type", "value", "value", "type", "type", "type", "type", None, None, None, "value"]
assert refers(program, point[2]) == ["type alias:Point"] and refers(program, point[3]) == ["variable:Point"]
assert refers(program, point[9]) == ["type alias:Point", "variable:Point"]  # an export names whatever it means
assert program.scope_of(point[10]) is None and program.scope_of(point[11]) is None  # exported, or from elsewhere
geo = names(tree, "Geo")
assert [program.space_of(n) for n in geo] == [None, "namespace", "value"]
assert refers(program, geo[1]) == ["namespace:Geo"] and refers(program, names(tree, "Shape")[1]) == []
assert refers(program, names(tree, "x")[1]) == ["parameter:guard.x"]
assert program.scope_of(next(n for n in F.walk(tree) if isinstance(n, S.Identifier) and n.name == "const")) is None
assert D.MEANINGS["class"] == {"value", "type", "namespace"} and D.MEANINGS["interface"] == {"type"}
print("ok")

## TSDEF-05 · Namespaces, ambient modules, `declare global` and `import a = b.c`

In [ ]:
tree, program = define("""
namespace Outer.Inner { export const deep = 1; }
namespace Outer { export namespace Inner { export const also = 2; } }
declare module "virtual" { export const v: number; }
declare module "short";
declare module "virtual" { export const w: number; }
declare global { interface Window { app: string } }
import Alias = Outer.Inner;
import Missing = Nowhere.Thing;
import Self = Self.X;
import Required = require("m");
const use = Alias.deep;
""")
assert listing(program)[:6] == [
    ("namespace", "Outer", 2), ("namespace", "Outer.Inner", 2), ("variable", "Outer.Inner.deep", 1),
    ("variable", "Outer.Inner.also", 1), ("module", "virtual", 2), ("variable", "virtual.v", 1)]
assert [e.kind for e in program.lookup("short")] == ["module"] and program.lookup("short")[0].scope.kind == "module"
assert program.lookup("Window")[0].kind == "interface"
alias, missing, self_, required = (program.lookup(name)[0] for name in ("Alias", "Missing", "Self", "Required"))
assert alias.target.qualified_name() == "Outer.Inner" and alias.resolved().kind == "namespace"
assert missing.target is None and self_.target is None and required.target is None
assert refers(program, names(tree, "Alias")[1]) == ["import:Alias"]
print("ok")

## TSDEF-06 · Types' own scopes: signatures, mapped types, `infer`, and JSX's components

In [ ]:
tree, program = define("""
enum Flags { A, // a comment
  [computed] = 1 }
type Mapped<T> = { [K in keyof T]: T[K] };
type Element<T> = T extends (infer E extends string)[] ? E : never;
type Fn = <U>(arg: U, ...rest: U[]) => U;
type Ctor = new (x: number) => object;
type Lit = { (a: string): void; new (b: string): Lit; m<V>(c: V): V; [k: string]: unknown };
""")
assert [(e.kind, e.name, e.parent.kind) for e in program.entities()] == [
    ("enum", "Flags", "module"), ("enum member", "A", "enum"), ("type alias", "Mapped", "module"),
    ("type parameter", "T", "type parameters"),
    ("type parameter", "K", "type parameters"), ("type alias", "Element", "module"),
    ("type parameter", "T", "type parameters"), ("type parameter", "E", "type parameters"),
    ("type alias", "Fn", "module"), ("type parameter", "U", "function"), ("parameter", "arg", "function"),
    ("parameter", "rest", "function"), ("type alias", "Ctor", "module"), ("parameter", "x", "function"),
    ("type alias", "Lit", "module"), ("parameter", "a", "function"), ("parameter", "b", "function"),
    ("type parameter", "V", "function"), ("parameter", "c", "function")]
assert refers(program, names(tree, "V")[-1]) == ["type parameter:V"]
assert refers(program, names(tree, "K")[1]) == ["type parameter:K"]
assert refers(program, names(tree, "E")[1]) == ["type parameter:E"]
assert refers(program, names(tree, "U")[-1]) == ["type parameter:U"]
assert refers(program, names(tree, "Lit")[1]) == ["type alias:Lit"]
assert program.scope_of(names(tree, "k")[0]) is None  # an index signature's parameter names nothing
tree, program = define("""
function Component() {}
const ns = { Item: Component };
const view = <div><Component /><ns.Item>text</ns.Item><svg:path d={ns} /></div>;
""", TypeScript59.JSX)
component = names(tree, "Component")
assert refers(program, names(tree, "ns")[-1]) == ["variable:ns"]
assert refers(program, component[2]) == ["function:Component"] and program.space_of(component[2]) == "value"
assert refers(program, names(tree, "ns")[1]) == ["variable:ns"] and program.scope_of(names(tree, "Item")[1]) is None
assert program.scope_of(names(tree, "div")[0]) is None and program.scope_of(names(tree, "svg")[0]) is None
print("ok")

## TSDEF-07 · Every entity of the conformance source, and every node located but the names of properties

In [ ]:
text = (SOURCES / "typescript59.ts").read_text(encoding="utf-8")
tree, program = define(text)
kinds = {}
for entity in program.entities():
    kinds[entity.kind] = kinds.get(entity.kind, 0) + 1
assert set(kinds) == set(D.MEANINGS), set(D.MEANINGS) - set(kinds)
unlocated = {type(n).__name__ for n in F.walk(tree) if program.scope_of(n) is None}
assert unlocated == {"Identifier", "PrivateIdentifier"}, unlocated
point2 = program.lookup("Point2")[0]
assert point2.kind == "import" and point2.target.qualified_name() == "Geometry.Point"
print(sum(kinds.values()), "entities", sorted(kinds.items()))